# Imports

In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import StratifiedShuffleSplit
from transformers import Wav2Vec2FeatureExtractor, AutoModel, get_cosine_schedule_with_warmup
import torchaudio.transforms as T
from collections import Counter
import matplotlib.pyplot as plt
import timm
import wandb

warnings.filterwarnings('ignore')

# Config

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
STEMS_DIR = Path('/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems')
MASHUP_DIR = Path('/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/mashups')
ESC50_DIR = Path('/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio')
TEST_CSV = Path('/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/test.csv')
OUT_DIR = Path('/kaggle/working/')

GENRES = ['blues','classical','country','disco','hiphop','jazz','metal','pop','reggae','rock']
GENRE2ID = {g: i for i, g in enumerate(GENRES)}
ID2GENRE = {i: g for g, i in GENRE2ID.items()}

MODEL_NAME = 'm-a-p/MERT-v1-95M'
SR = 24000
CLIP_SEC = 10.0
CLIP_SAMPLES = int(SR * CLIP_SEC)
BATCH_SIZE = 8
EPOCHS = 12
CNN_EPOCHS = 12
LR = 1e-4
WARMUP_RATIO = 0.1
VAL_SPLIT = 0.15
UNFREEZE_EPOCH = 2

SNR_MIN, SNR_MAX = 0, 15
NOISE_PROB = 0.6

OFFSETS = [0.0, 10.0, 20.0]
TTA_OFFSETS = [0.0, 2.0, 4.0, 6.0, 8.0, 10.0, 12.0, 15.0, 18.0, 21.0, 25.0]

CHECKPOINT = str(OUT_DIR / 'ast_v2_best.pt')

print(f'STEMS_DIR: {STEMS_DIR}')
print(f'MASHUP_DIR: {MASHUP_DIR}')
print(f'ESC50_DIR: {ESC50_DIR}')
print(f'TEST_CSV: {TEST_CSV}')
print(f'\nConfig: {EPOCHS} epochs | batch={BATCH_SIZE} | offsets={len(OFFSETS)} | TTA={len(TTA_OFFSETS)} clean + 4 noise')

# Weights and Biases

In [ ]:
wandb.login(key='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')

run = wandb.init(
    project='23f3002949-t12026',
    entity='ehsaasbhalla007-iit-madras',
    name='Final Notebook V5 MERT',
    config={
        'model': MODEL_NAME, 'epochs': EPOCHS, 'batch_size': BATCH_SIZE,
        'lr': LR, 'clip_sec': CLIP_SEC,
        'unfreeze_epoch': UNFREEZE_EPOCH,
        'snr_range': f'{SNR_MIN}-{SNR_MAX}dB',
        'noise_prob': NOISE_PROB,
        'n_train_offsets': len(OFFSETS),
        'n_tta_passes': len(TTA_OFFSETS) + 4,
        'spec_augment': True,
    },
    settings=wandb.Settings(start_method='thread'),
    resume='allow',
)
print(f'W&B run: {run.name}')

# Data Preparation

In [ ]:
def blend_stems(track_dir, sr=SR, duration=CLIP_SEC, offset=0.0):
    stems = list(Path(track_dir).glob('*.wav'))
    mixed = None
    start_sample = int(offset * sr)
    end_sample = start_sample + int(duration * sr)
    for sp in stems:
        try:
            key = str(sp)
            y, _ = librosa.load(key, sr=sr, mono=True, duration=duration, offset=offset)
            w = random.uniform(0.3, 1.5) if mixed is not None else 1.0
            mixed = y if mixed is None else mixed[:len(y)] + w * y[:len(mixed)]
        except Exception:
            pass
    if mixed is not None and np.max(np.abs(mixed)) > 0:
        mixed /= np.max(np.abs(mixed))
    return mixed


def overlay_noise(signal, noise, snr_db):
    n = np.tile(noise, int(np.ceil(len(signal)/len(noise))))[:len(signal)]
    ps = np.mean(signal**2) + 1e-9
    pn = np.mean(n**2) + 1e-9
    scale = np.sqrt(ps / pn / (10**(snr_db/10)))
    out = signal + scale * n
    if np.max(np.abs(out)) > 0:
        out /= np.max(np.abs(out))
    return out.astype(np.float32)

In [ ]:
feat_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)
spectrogram_transform = nn.Sequential(
    T.MelSpectrogram(sample_rate=SR, n_mels=128, n_fft=400, hop_length=160),
    T.AmplitudeToDB()
)

In [ ]:
noise_clips = []

if ESC50_DIR and ESC50_DIR.exists():
    esc50_files = list(ESC50_DIR.rglob('*.wav'))
    print(f'Loading {len(esc50_files)} ESC-50 clips at {SR}Hz...')
    for fp in tqdm(esc50_files, desc='ESC-50'):
        try:
            y, _ = librosa.load(str(fp), sr=SR, mono=True)
            if len(y) > 1000:
                noise_clips.append(y)
        except Exception:
            pass
    print(f'{len(noise_clips)} noise clips loaded')
else:
    print('ESC-50 not found, training without noise augmentation')

In [ ]:
all_entries = []
genre_tracks = {i: [] for i in range(len(GENRES))}
for genre in GENRES:
    gdir = STEMS_DIR / genre
    if not gdir.exists():
        print(f'  {gdir} not found, skipping')
        continue
    for track_dir in sorted(gdir.iterdir()):
        genre_tracks[GENRE2ID[genre]].append(str(track_dir))
        for off in OFFSETS:
            all_entries.append((str(track_dir), off, GENRE2ID[genre]))

random.shuffle(all_entries)
print(f'Total samples: {len(all_entries)}')
for gi, tracks in genre_tracks.items():
    print(f'  {ID2GENRE[gi]}: {len(tracks)} tracks')

labels_all = [s[2] for s in all_entries]
sss = StratifiedShuffleSplit(n_splits=1, test_size=VAL_SPLIT, random_state=SEED)
tr_idx, val_idx = next(sss.split(all_entries, labels_all))

train_split = [all_entries[i] for i in tr_idx]
val_split = [all_entries[i] for i in val_idx]
print(f'Train: {len(train_split)} | Val: {len(val_split)}')

# Exploratory Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

sample_picks = random.sample(range(len(all_entries)), 3)
for i, si in enumerate(sample_picks):
    track_dir, offset, label = all_entries[si]
    waveform = blend_stems(track_dir, offset=offset)
    if waveform is None:
        waveform = np.zeros(CLIP_SAMPLES, dtype=np.float32)
    if len(waveform) < CLIP_SAMPLES:
        waveform = np.pad(waveform, (0, CLIP_SAMPLES - len(waveform)))
    waveform = waveform[:CLIP_SAMPLES]

    axes[0][i].plot(waveform, linewidth=0.3, color='steelblue')
    axes[0][i].set_title(f'{ID2GENRE[label]} (offset={offset}s)')
    axes[0][i].set_ylim(-1.1, 1.1)

    mel = librosa.feature.melspectrogram(y=waveform.astype(np.float32), sr=SR, n_mels=128)
    axes[1][i].imshow(librosa.power_to_db(mel, ref=np.max), aspect='auto', origin='lower', cmap='magma')
    axes[1][i].set_title('Mel Spectrogram')

plt.tight_layout()
plt.show()

In [ ]:
genre_counts = Counter([ID2GENRE[s[2]] for s in all_entries])
genres_sorted = sorted(genre_counts.keys())

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(genres_sorted, [genre_counts[g] for g in genres_sorted], color='teal', edgecolor='black')
ax.set_title('Class Distribution')
ax.set_ylabel('Samples')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
durations = []
for genre in GENRES:
    gdir = STEMS_DIR / genre
    if not gdir.exists():
        continue
    for track_dir in list(gdir.iterdir())[:10]:
        for stem_file in list(track_dir.glob('*.wav'))[:1]:
            try:
                y, _ = librosa.load(str(stem_file), sr=SR, mono=True)
                durations.append(len(y) / SR)
            except Exception:
                pass

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(durations, bins=25, color='coral', edgecolor='black')
ax.axvline(CLIP_SEC, color='red', linestyle='--', label=f'Clip length ({CLIP_SEC}s)')
ax.set_title('Stem Duration Distribution')
ax.set_xlabel('Duration (seconds)')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

# Dataset

In [ ]:
def mask_spectrogram(spec, freq_mask_param=24, time_mask_param=80, n_masks=2):
    spec = spec.clone()
    T, F = spec.shape
    for _ in range(n_masks):
        f = random.randint(1, freq_mask_param)
        f0 = random.randint(0, max(F - f, 0))
        spec[:, f0:f0 + f] = 0.0
    for _ in range(n_masks):
        t = random.randint(1, time_mask_param)
        t0 = random.randint(0, max(T - t, 0))
        spec[t0:t0 + t, :] = 0.0
    return spec


class StemMixDataset(Dataset):
    def __init__(self, samples, feat_extractor, noise_clips=None, augment=True, genre_tracks=None):
        self.samples = samples
        self.feat_extractor = feat_extractor
        self.noise_clips = noise_clips or []
        self.augment = augment
        self.genre_tracks = genre_tracks or {}

    def __len__(self):
        return len(self.samples)

    def _cross_stem_blend(self, label, offset):
        tracks = self.genre_tracks.get(label, [])
        if len(tracks) < 2:
            return None
        stem_names = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
        start_sample = int(offset * SR)
        end_sample = start_sample + CLIP_SAMPLES
        mixed = None
        for sn in stem_names:
            td = random.choice(tracks)
            key = str(Path(td) / sn)
            try:
                s, _ = librosa.load(key, sr=SR, mono=True, duration=CLIP_SEC, offset=offset)
                w = random.uniform(0.3, 1.5) if mixed is not None else 1.0
                mixed = s if mixed is None else mixed[:len(s)] + w * s[:len(mixed)]
            except Exception:
                pass
        if mixed is not None and np.max(np.abs(mixed)) > 0:
            mixed /= np.max(np.abs(mixed))
        return mixed

    def __getitem__(self, idx):
        track_dir, offset, label = self.samples[idx]

        if self.augment and self.genre_tracks and random.random() < 0.5:
            y = self._cross_stem_blend(label, offset)
        else:
            y = None

        if y is None:
            y = blend_stems(track_dir, offset=offset)
        if y is None:
            y = np.zeros(CLIP_SAMPLES, dtype=np.float32)

        if len(y) < CLIP_SAMPLES:
            y = np.pad(y, (0, CLIP_SAMPLES - len(y)), mode='wrap')
        else:
            y = y[:CLIP_SAMPLES]
        if self.augment and random.random() < 0.3:
            rate = random.uniform(0.88, 1.12)
            y = librosa.effects.time_stretch(y, rate=rate)
            if len(y) > CLIP_SAMPLES:
                y = y[:CLIP_SAMPLES]
            else:
                y = np.pad(y, (0, CLIP_SAMPLES - len(y)), mode='wrap')

        if self.augment and self.noise_clips and random.random() < NOISE_PROB:
            y = overlay_noise(y, random.choice(self.noise_clips), random.uniform(SNR_MIN, SNR_MAX))

        if self.augment and random.random() > 0.5:
            y = np.roll(y, random.randint(0, CLIP_SAMPLES // 4))

        inputs = self.feat_extractor(y.astype(np.float32), sampling_rate=SR, return_tensors='pt')
        raw_audio = inputs['input_values'].squeeze(0)
        
        cnn_spec = spectrogram_transform(torch.tensor(y.astype(np.float32))).transpose(0, 1)

        if self.augment:
            cnn_spec = mask_spectrogram(cnn_spec)

        return {'raw_audio': raw_audio, 'cnn_spec': cnn_spec, 'labels': torch.tensor(label, dtype=torch.long)}

# Dataloader

In [ ]:
train_set = StemMixDataset(train_split, feat_extractor, noise_clips, augment=True, genre_tracks=genre_tracks)
val_set = StemMixDataset(val_split, feat_extractor, noise_clips, augment=False)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

v1_f1 = [0.9890, 0.9890, 0.9556, 0.9773, 0.9091, 0.9888, 0.9149, 0.9362, 0.9655, 0.8636]
raw_w = [1.0 / f for f in v1_f1]
mean_w = sum(raw_w) / len(raw_w)
class_weights = torch.tensor([w / mean_w for w in raw_w], dtype=torch.float32).to(DEVICE)

sample = train_set[0]
print(f'Train batches: {len(train_loader)}')
print(f'Val batches: {len(val_loader)}')
print(f'Input shape: {sample["raw_audio"].shape}')
print(f'Class weights: {[round(float(w), 3) for w in class_weights]}')

# Models

In [ ]:
print(f'Loading: {MODEL_NAME}')

class MERTClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.mert = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True)
        self.mert.gradient_checkpointing_enable()
        self.classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, raw_audio):
        outputs = self.mert(raw_audio)
        hidden = outputs.last_hidden_state.mean(dim=1)
        return self.classifier(hidden)

model = MERTClassifier(num_classes=10).to(DEVICE)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'MERT loaded | Total: {total:,} | Trainable: {trainable:,}')

In [ ]:
class SpectrogramClassifierB2(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b2', pretrained=True, num_classes=num_classes, in_chans=1)

    def forward(self, x):
        return self.backbone(x)

In [ ]:
class CompactAudioCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.head(self.features(x))

In [ ]:
class SpectrogramDenseNet121(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.backbone = timm.create_model('densenet121', pretrained=True, num_classes=num_classes, in_chans=1)

    def forward(self, x):
        return self.backbone(x)

# Training

In [ ]:
def mixup_data(x, y, alpha=0.2):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(DEVICE)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

def set_encoder_trainable(model, trainable):
    for name, param in model.named_parameters():
        if 'classifier' not in name:
            param.requires_grad = trainable

set_encoder_trainable(model, False)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=0.01
)
total_steps = EPOCHS * len(train_loader)
scheduler = get_cosine_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total_steps), total_steps)
loss_fn = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

print(f'Total steps: {total_steps} | Warmup: {int(WARMUP_RATIO * total_steps)}')

In [ ]:
peak_f1 = 0.0
epoch_log = []
START_EPOCH = 1

if Path(CHECKPOINT).exists():
    print(f'Found checkpoint: {CHECKPOINT}')
    ckpt = torch.load(CHECKPOINT, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    peak_f1 = ckpt['best_val_f1']
    epoch_log = ckpt.get('history', [])
    START_EPOCH = ckpt['epoch'] + 1
    print(f'   Resuming from epoch {START_EPOCH} | Best val F1: {peak_f1:.4f}')

    if START_EPOCH > UNFREEZE_EPOCH:
        set_encoder_trainable(model, True)
        optimizer = torch.optim.AdamW([
            {'params': model.mert.parameters(), 'lr': LR * 0.1},
            {'params': model.classifier.parameters(), 'lr': LR},
        ], weight_decay=0.01)
        remaining = max(1, (EPOCHS - START_EPOCH + 1)) * len(train_loader)
        scheduler = get_cosine_schedule_with_warmup(optimizer, 0, remaining)
        print(f'   Encoder unfrozen | remaining steps: {remaining}')
else:
    print('Starting fresh training...')

In [ ]:
for epoch in range(START_EPOCH, EPOCHS + 1):

    if epoch == UNFREEZE_EPOCH:
        print(f'  Epoch {epoch}: Unfreezing full encoder')
        set_encoder_trainable(model, True)
        optimizer = torch.optim.AdamW([
            {'params': model.mert.parameters(), 'lr': LR * 0.1},
            {'params': model.classifier.parameters(), 'lr': LR},
        ], weight_decay=0.01)
        remaining = (EPOCHS - epoch + 1) * len(train_loader)
        scheduler = get_cosine_schedule_with_warmup(optimizer, 0, remaining)

    model.train()
    train_loss, train_preds, train_labels = 0.0, [], []
    for batch in tqdm(train_loader, desc=f'Ep {epoch:02d} Train', leave=False):
        iv, labels = batch['raw_audio'].to(DEVICE), batch['labels'].to(DEVICE)
        iv, y_a, y_b, lam = mixup_data(iv, labels)
        optimizer.zero_grad()
        out = model(raw_audio=iv)
        loss = mixup_criterion(loss_fn, out, y_a, y_b, lam)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()
        train_preds += out.argmax(-1).cpu().tolist()
        train_labels += labels.cpu().tolist()
    train_f1 = f1_score(train_labels, train_preds, average='macro')

    model.eval()
    val_loss, val_preds, val_labels = 0.0, [], []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'Ep {epoch:02d} Val  ', leave=False):
            iv, labels = batch['raw_audio'].to(DEVICE), batch['labels'].to(DEVICE)
            out = model(raw_audio=iv)
            loss = loss_fn(out, labels)
            val_loss += loss.item()
            val_preds += out.argmax(-1).cpu().tolist()
            val_labels += labels.cpu().tolist()
    val_f1 = f1_score(val_labels, val_preds, average='macro')

    log = {
        'epoch': epoch,
        'train_loss': train_loss / len(train_loader),
        'val_loss': val_loss / len(val_loader),
        'train_f1': train_f1,
        'val_f1': val_f1,
        'lr': optimizer.param_groups[-1]['lr'],
    }
    epoch_log.append(log)
    wandb.log(log)

    marker = ' *' if val_f1 > peak_f1 else ''
    tl, vl = log['train_loss'], log['val_loss']
    print(f'Ep {epoch:02d} | tr_loss={tl:.4f} tr_f1={train_f1:.4f} | va_loss={vl:.4f} va_f1={val_f1:.4f}{marker}')

    if val_f1 > peak_f1:
        peak_f1 = val_f1
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_val_f1': peak_f1,
            'history': epoch_log,
        }, CHECKPOINT)

print(f'\nBest Val Macro F1: {peak_f1:.4f}')

# Comparison Models

In [ ]:
model.cpu()
torch.cuda.empty_cache()
print(f'MERT moved to CPU, GPU memory freed for CNN training')

CNN_LR = 1e-3

def train_cnn_model(cnn_model, name, train_loader, val_loader, epochs=CNN_EPOCHS):
    cnn_model = cnn_model.to(DEVICE)
    cnn_optimizer = torch.optim.AdamW(cnn_model.parameters(), lr=CNN_LR, weight_decay=0.01)
    cnn_loss_fn = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
    cnn_best_f1 = 0.0

    for ep in range(1, epochs + 1):
        cnn_model.train()
        running_loss = 0.0
        for batch in train_loader:
            x = batch['cnn_spec'].unsqueeze(1).to(DEVICE)
            y = batch['labels'].to(DEVICE)
            if x.size(2) > 256:
                start = random.randint(0, x.size(2) - 256)
                x = x[:, :, start:start+256, :]
            x, y_a, y_b, lam = mixup_data(x, y)
            cnn_optimizer.zero_grad()
            out = cnn_model(x)
            loss = mixup_criterion(cnn_loss_fn, out, y_a, y_b, lam)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(cnn_model.parameters(), 1.0)
            cnn_optimizer.step()
            running_loss += loss.item()

        cnn_model.eval()
        cnn_preds, cnn_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                x = batch['cnn_spec'].unsqueeze(1).to(DEVICE)
                y = batch['labels'].to(DEVICE)
                out = cnn_model(x)
                cnn_preds += out.argmax(-1).cpu().tolist()
                cnn_labels += y.cpu().tolist()

        ep_f1 = f1_score(cnn_labels, cnn_preds, average='macro')
        ep_loss = running_loss / len(train_loader)
        if ep_f1 > cnn_best_f1:
            cnn_best_f1 = ep_f1
            torch.save(cnn_model.state_dict(), str(OUT_DIR / f'{name}_best.pt'))

        wandb.log({f'{name}_epoch': ep, f'{name}_loss': ep_loss, f'{name}_val_f1': ep_f1})
        print(f'[{name}] Ep {ep:02d} | loss={ep_loss:.4f} | val_f1={ep_f1:.4f}')

    print(f'[{name}] Best F1: {cnn_best_f1:.4f}')
    return cnn_best_f1

In [ ]:
print('Training EfficientNet-B2...')
effnet_f1 = train_cnn_model(SpectrogramClassifierB2(num_classes=10), 'effnet_b2', train_loader, val_loader)

In [ ]:
print('Training DenseNet121...')
dense_f1 = train_cnn_model(SpectrogramDenseNet121(num_classes=10), 'densenet_121', train_loader, val_loader)

In [ ]:
print('Training Custom CNN (From Scratch)...')
compact_f1 = train_cnn_model(CompactAudioCNN(num_classes=10), 'compact_cnn', train_loader, val_loader)

# Validation

In [ ]:
hist_df = pd.DataFrame(epoch_log)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(hist_df.epoch, hist_df.train_loss, 'o-', label='Train')
axes[0].plot(hist_df.epoch, hist_df.val_loss, 'o-', label='Val')
axes[0].set_title('Loss')
axes[0].legend()
axes[1].plot(hist_df.epoch, hist_df.train_f1, 'o-', label='Train')
axes[1].plot(hist_df.epoch, hist_df.val_f1, 'o-', label='Val')
axes[1].axhline(0.8, color='lime', ls='--', label='0.80 target')
axes[1].set_title('Macro F1')
axes[1].legend()
plt.tight_layout()
plt.savefig(str(OUT_DIR / 'ast_v2_curves.png'), dpi=120)
plt.show()

In [ ]:
torch.cuda.empty_cache()
ckpt = torch.load(CHECKPOINT, map_location=DEVICE)
model = model.to(DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

eval_preds, eval_labels = [], []
with torch.no_grad():
    for batch in tqdm(val_loader, desc='Final eval'):
        iv = batch['raw_audio'].to(DEVICE)
        lbl = batch['labels'].to(DEVICE)
        eval_preds += model(raw_audio=iv).argmax(-1).cpu().tolist()
        eval_labels += lbl.cpu().tolist()

print(classification_report(eval_labels, eval_preds, target_names=GENRES, digits=4))

# Ensemble Strategy Data Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.bar(['MERT-v1', 'DenseNet121', 'EffNetB2'], [peak_f1, dense_f1, effnet_f1], color=['purple', 'teal', 'coral'])
ax.set_title('Peak Validation F1 by Model (Standalone run)')
ax.set_ylim(0.7, 1.0)
for i, v in enumerate([peak_f1, dense_f1, effnet_f1]):
    ax.text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

# Inference

In [ ]:
class InferenceDataset(Dataset):
    def __init__(self, paths, feat_extractor, offset=0.0, noise_clips=None, snr_db=None):
        self.paths = paths
        self.fe = feat_extractor
        self.offset = offset
        self.noise_clips = noise_clips or []
        self.snr_db = snr_db

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        try:
            y, _ = librosa.load(str(self.paths[idx]), sr=SR, mono=True, duration=CLIP_SEC, offset=self.offset)
            if len(y) < CLIP_SAMPLES:
                y = np.pad(y, (0, CLIP_SAMPLES - len(y)), mode='wrap')
            y = y[:CLIP_SAMPLES]
            if np.max(np.abs(y)) > 0:
                y = y / np.max(np.abs(y))
        except Exception:
            y = np.zeros(CLIP_SAMPLES, dtype=np.float32)

        if self.snr_db is not None and self.noise_clips:
            y = overlay_noise(y, random.choice(self.noise_clips), self.snr_db)

        raw_audio = self.fe(y.astype(np.float32), sampling_rate=SR, return_tensors='pt')['input_values'].squeeze(0)
        return {'raw_audio': raw_audio}

In [ ]:
test_df = pd.read_csv(TEST_CSV)
test_paths = [MASHUP_DIR / Path(row['filename']).name for _, row in test_df.iterrows()]
test_ids = test_df['id'].tolist()

prob_accumulator = np.zeros((len(test_ids), 10), dtype=np.float64)
total_passes = 0

for tta_off in TTA_OFFSETS:
    print(f'Clean TTA | offset={tta_off}s')
    ds = InferenceDataset(test_paths, feat_extractor, offset=tta_off)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
    ptr = 0
    with torch.no_grad():
        for batch in tqdm(dl):
            v = batch['raw_audio'].to(DEVICE)
            probs = torch.softmax(model(raw_audio=v), -1).cpu().numpy()
            prob_accumulator[ptr:ptr+len(probs)] += probs
            ptr += len(probs)
    total_passes += 1

if noise_clips:
    for snr in [8, 12]:
        for tta_off in [0.0, 10.0]:
            print(f'Noise TTA | offset={tta_off}s | SNR={snr}dB')
            ds = InferenceDataset(test_paths, feat_extractor, offset=tta_off, noise_clips=noise_clips, snr_db=snr)
            dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
            ptr = 0
            with torch.no_grad():
                for batch in tqdm(dl):
                    v = batch['raw_audio'].to(DEVICE)
                    probs = torch.softmax(model(raw_audio=v), -1).cpu().numpy()
                    prob_accumulator[ptr:ptr+len(probs)] += probs
                    ptr += len(probs)
            total_passes += 1
else:
    print('No noise pool, skipping noise TTA')

prob_accumulator /= total_passes
predicted_genres = [ID2GENRE[i] for i in prob_accumulator.argmax(axis=1)]

print(f'\nTotal TTA passes: {total_passes}')
print('\nPrediction distribution:')
pc = Counter(predicted_genres)
for g in GENRES:
    print(f'  {g:12s}: {pc.get(g,0):5d}  ({100*pc.get(g,0)/len(predicted_genres):.1f}%)')

# Submission

In [ ]:
submission = pd.DataFrame({
    'id': [str(i).zfill(4) for i in test_ids],
    'genre': predicted_genres,
})
sub_path = str(OUT_DIR / 'submission_ast_v2.csv')
submission.to_csv(sub_path, index=False)
print(f'Saved: {sub_path}')
print(submission.head())

In [ ]:
wandb.log({'best_val_macro_f1': peak_f1})
wandb.log({'best_effnet_b2_f1': effnet_f1, 'best_densenet_f1': dense_f1, 'best_compact_cnn_f1': compact_f1})
per_class = f1_score(eval_labels, eval_preds, average=None)
wandb.log({f'val_f1_{g}': float(per_class[i]) for i, g in enumerate(GENRES)})
wandb.log({'tta_passes': total_passes})
wandb.log({'curves': wandb.Image(str(OUT_DIR / 'ast_v2_curves.png'))})
wandb.save(sub_path)
wandb.finish()
print('W&B logged and finished')